# Inventario y balance de los datos

Comprueba que los datos están completos antes de empezar: cuántos ficheros hay
en cada conjunto, cuántos registros y qué periodo cubren.

Los datos se descargan del enlace de la entrega y se descomprimen en `OUTPUTS/`;
el `README.md` explica dónde. Este cuaderno no descarga nada.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from tfm import ejecutar, rutas
ejecutar.situacion()

## Qué hay en `OUTPUTS/`

Si esta celda no encuentra nada, revisa el apartado «Disponer los datos» del `README.md`.

In [ ]:
import glob, os
from collections import Counter

def inventario(base):
    filas = []
    for carpeta in sorted(glob.glob(os.path.join(base, "*"))):
        if not os.path.isdir(carpeta):
            continue
        ficheros = glob.glob(os.path.join(carpeta, "*"))
        tam = sum(os.path.getsize(f) for f in ficheros if os.path.isfile(f))
        filas.append({"conjunto": os.path.basename(carpeta),
                      "ficheros": len(ficheros),
                      "GB": round(tam / 1024**3, 2)})
    return filas

import pandas as pd
for etiqueta, base in [("bronze (JSONL)", rutas.bronze()),
                       ("OUTPUTS/gold",   rutas.parquet())]:
    filas = inventario(base)
    print(f"\n{etiqueta}: {os.path.dirname(base) if base.endswith('*.parquet') else base}")
    display(pd.DataFrame(filas) if filas else "vacío")

## Balance de registros

Cuenta sobre los parquet, que son la fuente de verdad del pipeline.

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("SET memory_limit='6GB'")
for tabla in ("contrato_2025", "nodo_2025", "vinculo_2025"):
    patron = rutas.parquet(tabla).replace("\\", "/")
    try:
        n = con.execute(f"SELECT count(*) FROM read_parquet('{patron}')").fetchone()[0]
        print(f"{tabla:16} {n:>12,}")
    except Exception as e:
        print(f"{tabla:16} no disponible ({type(e).__name__})")
con.close()

## Integridad del modelo

Comprobaciones de coherencia sobre las tablas: unicidad de claves, registros huérfanos y rango temporal.

In [ ]:
ejecutar.correr('tfm.verificacion.invariantes')